# Medical Record Summarization

Maintained by [viraj54](https://github.com/viraj54).

A notebook experiment comparing extractive TF-IDF, pretrained BART and BART fine-tuning on clinical-note targets. Run in a dedicated Python 3.11 environment using the repository requirements. Recorded aggregate results are in `results/`; this output-cleared notebook has not been retrained during repository preparation.

**Research demonstration:** generated text and lexical metrics do not establish factual correctness. Do not use outputs for patient-care decisions. Training may require substantial GPU memory. Dataset/model downloads occur during execution.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / "outputs"
ARTIFACT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_ROOT)
print("Artifacts will be saved to:", ARTIFACT_DIR)


## Section 1 — Install and Import

In [ ]:
# Install requirements.txt in a dedicated Python 3.11 environment before running.
# This notebook does not modify an existing Kaggle/Jupyter environment automatically.
import transformers, accelerate
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)


In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import nltk

from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, pipeline,
    EarlyStoppingCallback, set_seed
)
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings("default")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | Count: {torch.cuda.device_count()}")
print("Imports done.")

## Section 2 — Load Dataset and EDA

In [ ]:
print("Loading dataset...")
raw_dataset = load_dataset("AGBonnet/augmented-clinical-notes")
df = raw_dataset["train"].to_pandas()
print(f"Shape: {df.shape} | Columns: {df.columns.tolist()}")
print("\nNull check:")
print(df[["note","summary"]].isnull().sum())

In [ ]:
df["note_wlen"] = df["note"].fillna("").apply(lambda x: len(x.split()))
df["summary_wlen"] = df["summary"].fillna("").apply(lambda x: len(x.split()))
df["compression_ratio"] = df["summary_wlen"] / (df["note_wlen"] + 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df["note_wlen"].clip(upper=1000), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Note Word Length"); axes[0].set_xlabel("Words")
axes[1].hist(df["summary_wlen"].clip(upper=600), bins=50, color="darkorange", edgecolor="white")
axes[1].set_title("Summary Word Length"); axes[1].set_xlabel("Words")
axes[2].hist(df["compression_ratio"].clip(upper=5), bins=50, color="mediumseagreen", edgecolor="white")
axes[2].set_title("Compression Ratio"); axes[2].set_xlabel("Ratio")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Median note words: {df['note_wlen'].median():.0f}")
print(f"Median summary words: {df['summary_wlen'].median():.0f}")
print(f"Median compression ratio: {df['compression_ratio'].median():.3f}")

## Section 3 — Preprocessing

The experiment samples 5,000 rows and uses a 70/15/15 row-level split with seed 42. Input/target token limits are 512/384. The final comparison uses the first 50 test rows. These settings do not establish patient-level separation or eliminate duplicated cases.


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────
MODEL_NAME     = "facebook/bart-large-cnn"   # UPGRADE: large-cnn > base
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 384    # UPGRADE: 384 > 256 — summaries are long JSON
N_SAMPLES      = 5000   # UPGRADE: 5000 > 3000
N_EVAL         = 50
# ────────────────────────────────────────────────────────────────────────

def clean_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r"\s+", " ", text).strip()

df_clean = df[["note","summary"]].dropna().copy()
df_clean["note"]    = df_clean["note"].apply(clean_text)
df_clean["summary"] = df_clean["summary"].apply(clean_text)
df_clean = df_clean[(df_clean["note"].str.len() > 10) & (df_clean["summary"].str.len() > 10)]

if len(df_clean) < N_SAMPLES:
    raise ValueError("Not enough valid dataset rows for the configured sample size")
df_sampled = df_clean.sample(n=N_SAMPLES, random_state=SEED).reset_index(drop=True)
train_df, temp_df  = train_test_split(df_sampled, test_size=0.30, random_state=SEED)
val_df,   test_df  = train_test_split(temp_df,    test_size=0.50, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

test_notes    = test_df["note"].tolist()[:N_EVAL]
test_summaries = test_df["summary"].tolist()[:N_EVAL]

In [ ]:
# Convert to HuggingFace DatasetDict
dataset_dict = DatasetDict({
    "train":      Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test":       Dataset.from_pandas(test_df)
})

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    inputs = tokenizer(
        examples["note"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        targets = tokenizer(
            examples["summary"],
            max_length=MAX_TARGET_LEN,
            truncation=True,
            padding="max_length"
        )
    labels = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in targets["input_ids"]
    ]
    inputs["labels"] = labels
    return inputs

print("Tokenizing...")
tokenized_datasets = dataset_dict.map(
    tokenize_function, batched=True,
    remove_columns=["note","summary"], desc="Tokenizing"
)
print("Done.")
print(tokenized_datasets)

## Section 4 — Extractive Baseline (TF-IDF, no dependency issues)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_extractive_summary(text, n_sentences=5):
    """TF-IDF extractive summarizer using scikit-learn."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.split()) > 4]
    if len(sentences) <= n_sentences:
        return " ".join(sentences)
    try:
        tfidf = TfidfVectorizer(stop_words='english')
        tfidf_matrix = tfidf.fit_transform(sentences)
        doc_vector = np.asarray(tfidf_matrix.mean(axis=0))
        scores = cosine_similarity(tfidf_matrix, doc_vector).flatten()
        top_idx = sorted(np.argsort(scores)[-n_sentences:].tolist())
        return " ".join([sentences[i] for i in top_idx])
    except Exception:
        return " ".join(sentences[:n_sentences])

print("Running TF-IDF extractive summarization on 50 test samples...")
extractive_preds = []
extractive_refs  = []
for i, note in enumerate(test_notes):
    extractive_preds.append(tfidf_extractive_summary(note, n_sentences=5))
    extractive_refs.append(test_summaries[i])

print("Done.")
print("\nSAMPLE EXTRACTIVE OUTPUT:")
print(extractive_preds[0][:400])

## Section 5 — Pre-trained BART (Zero-shot)

In [ ]:
print("Loading facebook/bart-large-cnn for zero-shot inference...")
try:
    pretrained_bart_pipeline = pipeline(
        "summarization", model="facebook/bart-large-cnn",
        device=0 if DEVICE=="cuda" else -1, truncation=True
    )
    print("Loaded.")
except Exception as e:
    print(f"Error: {e}"); pretrained_bart_pipeline = None

pretrained_preds = []
pretrained_refs  = []
if pretrained_bart_pipeline:
    print("Running zero-shot inference...")
    for i, note in enumerate(test_notes):
        try:
            r = pretrained_bart_pipeline(note[:1024], max_length=256, min_length=60, do_sample=False, truncation=True)
            pretrained_preds.append(r[0]["summary_text"])
        except Exception as e:
            pretrained_preds.append("Error.")
        pretrained_refs.append(test_summaries[i])
        if (i+1) % 10 == 0: print(f"  {i+1}/{N_EVAL}")
    print("Done.")
else:
    pretrained_preds = ["unavailable"] * N_EVAL
    pretrained_refs  = test_summaries[:N_EVAL]

print("\nSAMPLE PRE-TRAINED BART OUTPUT:")
print(pretrained_preds[0])

## Section 6 — Fine-tune BART-large-CNN

Up to five epochs, validation-loss checkpoint selection and early-stopping patience 2. The nominal per-device batch is 2 with four accumulation steps; global effective batch size also depends on the number of devices.


In [ ]:
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"Parameters: {model.num_parameters():,}")

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if DEVICE=="cuda" else None
)

OUTPUT_DIR = str(ARTIFACT_DIR / "bart_large_medical_summarizer")

total_steps = (len(tokenized_datasets["train"]) // (4 * 2)) * 5  # batch*grad_accum * epochs
warmup_steps = int(0.06 * total_steps)  # 6% warmup
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,              # slightly higher for large model
    per_device_train_batch_size=2,   # large model needs smaller batch
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",      # cosine decay
    label_smoothing_factor=0.1,      # label smoothing reduces overconfidence
    predict_with_generate=True,
    generation_max_length=384,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    seed=SEED
)

print("Training args configured.")

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # stop if no improvement for 2 epochs
)

print("Starting fine-tuning...")
print(f"Train: {len(tokenized_datasets['train'])} | Val: {len(tokenized_datasets['validation'])}")
print("=" * 60)

train_result = trainer.train()

print("\nFINE-TUNING COMPLETE!")
print(f"Runtime: {train_result.metrics.get('train_runtime', 0):.1f}s")
print(f"Train loss: {train_result.metrics.get('train_loss', 0):.4f}")

In [ ]:
FINAL_MODEL_DIR = str(ARTIFACT_DIR / "bart_large_medical_summarizer_final")
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# Plot training loss curve
log_history = trainer.state.log_history
train_logs = [(x["step"], x["loss"]) for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

fig, ax = plt.subplots(figsize=(11, 5))
if train_logs:
    ax.plot([x[0] for x in train_logs], [x[1] for x in train_logs],
            color="steelblue", lw=2, label="Train Loss", alpha=0.85)
if eval_logs:
    ax.plot([x[0] for x in eval_logs], [x[1] for x in eval_logs],
            color="darkorange", lw=2.5, marker="o", ms=7, label="Val Loss")
ax.set_title("Fine-tuned BART-large-cnn: Loss Curve", fontsize=14, fontweight="bold")
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_loss_curve.png")

## Section 7 — Generate Summaries with Fine-tuned Model

In [ ]:
print(f"Loading fine-tuned model from {FINAL_MODEL_DIR}...")
try:
    finetuned_pipeline = pipeline(
        "summarization", model=FINAL_MODEL_DIR,
        tokenizer=FINAL_MODEL_DIR,
        device=0 if DEVICE=="cuda" else -1,
        truncation=True
    )
    print("Pipeline loaded.")
except Exception as e:
    print(f"Error: {e}"); finetuned_pipeline = None

finetuned_preds = []
finetuned_refs  = []
if finetuned_pipeline:
    print("Running fine-tuned inference on 50 samples...")
    for i, note in enumerate(test_notes):
        try:
            r = finetuned_pipeline(
                note, max_length=384, min_length=80,
                do_sample=False, truncation=True,
                num_beams=4,          # beam search for better quality
                length_penalty=1.2,   # encourage longer, complete summaries
                no_repeat_ngram_size=3
            )
            finetuned_preds.append(r[0]["summary_text"])
        except Exception as e:
            finetuned_preds.append("Error.")
        finetuned_refs.append(test_summaries[i])
        if (i+1) % 10 == 0: print(f"  {i+1}/{N_EVAL}")
    print("Done.")
else:
    finetuned_preds = ["unavailable"] * N_EVAL
    finetuned_refs  = test_summaries[:N_EVAL]

print("\nSAMPLE FINE-TUNED OUTPUT:")
print(finetuned_preds[0])

## Section 8 — ROUGE Evaluation

In [ ]:
def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not ref: continue
        try:
            s = scorer.score(ref, pred)
            r1.append(s["rouge1"].fmeasure)
            r2.append(s["rouge2"].fmeasure)
            rL.append(s["rougeL"].fmeasure)
        except: continue
    return {"ROUGE-1": np.mean(r1) if r1 else 0.0,
            "ROUGE-2": np.mean(r2) if r2 else 0.0,
            "ROUGE-L": np.mean(rL) if rL else 0.0}

print("Computing ROUGE scores...")
ext_rouge = compute_rouge(extractive_preds, extractive_refs)
pre_rouge = compute_rouge(pretrained_preds, pretrained_refs)
ft_rouge  = compute_rouge(finetuned_preds,  finetuned_refs)

rouge_df = pd.DataFrame([
    {"Model": "Extractive (TF-IDF)", **ext_rouge},
    {"Model": "Pre-trained BART",    **pre_rouge},
    {"Model": "Fine-tuned BART",     **ft_rouge}
])
for c in ["ROUGE-1","ROUGE-2","ROUGE-L"]:
    rouge_df[c] = rouge_df[c].round(4)

print("\nROUGE RESULTS:")
print(rouge_df.to_string(index=False))

## Section 9 — Additional evaluation

BERTScore and METEOR compare generated text with reference summaries. Entity extraction uses SciSpaCy if separately installed; otherwise it uses a regex keyword fallback, as in the recorded run. Source-token overlap and lexicon concept F1 are custom lexical heuristics, not factual verification or official MEDCON/UMLS linking. Negation, drug doses and clinical relationships require separate assessment.


In [ ]:
# ── METRIC 1: BERTScore ─────────────────────────────────────────────────
print("Computing BERTScore (this may take ~2-3 minutes)...")

def compute_bertscore(predictions, references, model_type="distilbert-base-uncased"):
    """Compute average BERTScore F1."""
    valid_preds = [p if p and p.strip() else "empty" for p in predictions]
    valid_refs  = [r if r and r.strip() else "empty" for r in references]
    try:
        P, R, F1 = bert_score_fn(
            valid_preds, valid_refs,
            model_type=model_type,
            lang="en", verbose=False,
            device=DEVICE
        )
        return float(F1.mean())
    except Exception as e:
        print(f"  BERTScore error: {e}")
        return 0.0

ext_bs = compute_bertscore(extractive_preds, extractive_refs)
print(f"  Extractive BERTScore F1: {ext_bs:.4f}")
pre_bs = compute_bertscore(pretrained_preds, pretrained_refs)
print(f"  Pre-trained BART BERTScore F1: {pre_bs:.4f}")
ft_bs  = compute_bertscore(finetuned_preds, finetuned_refs)
print(f"  Fine-tuned BART BERTScore F1: {ft_bs:.4f}")
print("BERTScore done.")

In [ ]:
# ── METRIC 2: METEOR ─────────────────────────────────────────────────────
print("Computing METEOR scores...")

def compute_meteor(predictions, references):
    scores = []
    for pred, ref in zip(predictions, references):
        if not pred or not ref: continue
        try:
            pred_tokens = word_tokenize(pred.lower())
            ref_tokens  = word_tokenize(ref.lower())
            scores.append(meteor_score([ref_tokens], pred_tokens))
        except: continue
    return np.mean(scores) if scores else 0.0

ext_meteor = compute_meteor(extractive_preds, extractive_refs)
pre_meteor = compute_meteor(pretrained_preds, pretrained_refs)
ft_meteor  = compute_meteor(finetuned_preds,  finetuned_refs)
print(f"  Extractive METEOR:    {ext_meteor:.4f}")
print(f"  Pre-trained METEOR:   {pre_meteor:.4f}")
print(f"  Fine-tuned METEOR:    {ft_meteor:.4f}")

In [ ]:
# ── METRIC 3: Medical Keyword preservation ──────────────────────────────────
# Uses scispacy's en_core_sci_sm model to extract medical entities
# from reference summaries, then checks how many appear in predictions.
print("Computing Medical Keyword preservation scores...")

try:
    import spacy
    nlp_sci = spacy.load("en_core_sci_sm")
    SCISPACY_AVAILABLE = True
    print("  scispacy model loaded.")
except Exception as e:
    print(f"  scispacy not available ({e}). Using keyword-based fallback.")
    SCISPACY_AVAILABLE = False

# Core medical entity keywords used as fallback
MEDICAL_ENTITY_PATTERNS = [
    r'\b(diagnosis|diagnos\w+)\b',
    r'\b(treatment|treat\w+)\b',
    r'\b(symptom\w*)\b',
    r'\b(medication\w*|drug\w*|prescription)\b',
    r'\b(patient|hospital|clinic\w*)\b',
    r'\b(blood pressure|heart rate|temperature|oxygen)\b',
    r'\b(mg|ml|mcg|mmhg|bpm)\b',
    r'\b(surgery|surgical|operation|procedure)\b',
    r'\b(cancer|tumor|malignant|benign)\b',
    r'\b(test|exam\w*|scan|mri|ct|xray|ultrasound|biopsy)\b',
    r'\b(discharge|admission|emergency)\b',
    r'\b(pain|fever|nausea|fatigue|inflammation)\b'
]

def extract_medical_entities_spacy(text):
    """Extract medical entities using scispacy."""
    doc = nlp_sci(text[:5000])  # limit for speed
    return set([ent.text.lower().strip() for ent in doc.ents if len(ent.text.strip()) > 2])

def extract_medical_entities_regex(text):
    """Fallback: extract medical keywords using regex patterns."""
    text_lower = text.lower()
    entities = set()
    for pattern in MEDICAL_ENTITY_PATTERNS:
        matches = re.findall(pattern, text_lower)
        entities.update(matches)
    return entities

def extract_medical_entities(text):
    if SCISPACY_AVAILABLE:
        return extract_medical_entities_spacy(text)
    return extract_medical_entities_regex(text)

def compute_ner_preservation(predictions, references):
    """
    For each sample: extract medical entities from reference,
    check what % appear in prediction.
    Returns average preservation score across all samples.
    """
    scores = []
    for pred, ref in zip(predictions, references):
        if not pred or not ref: continue
        try:
            ref_entities  = extract_medical_entities(ref)
            if not ref_entities:
                continue
            pred_text_lower = pred.lower()
            preserved = sum(1 for ent in ref_entities if ent in pred_text_lower)
            scores.append(preserved / len(ref_entities))
        except: continue
    return np.mean(scores) if scores else 0.0

ext_ner = compute_ner_preservation(extractive_preds, extractive_refs)
pre_ner = compute_ner_preservation(pretrained_preds, pretrained_refs)
ft_ner  = compute_ner_preservation(finetuned_preds,  finetuned_refs)
print(f"  Extractive Keyword preservation:    {ext_ner:.4f}")
print(f"  Pre-trained BART Keyword preservation: {pre_ner:.4f}")
print(f"  Fine-tuned BART Keyword preservation:  {ft_ner:.4f}")

In [ ]:
# ── METRIC 4: Source-token overlap Score ─────────────────────────────────
# Measures what fraction of clinical facts in the generated summary
# can be traced back to the source note (not just the reference).
# Uses token-level precision: for each sentence in the prediction,
# compute how many content tokens appear in the source note.
# This approximates faithfulness / factual grounding without an NLI model.
print("Computing Source-token overlap Scores...")

import string

STOPWORDS = {
    "the","a","an","is","was","are","were","be","been","being",
    "have","has","had","do","does","did","will","would","could",
    "should","may","might","shall","can","of","in","on","at","to",
    "for","with","by","from","as","and","or","but","not","no",
    "this","that","it","its","he","she","they","we","i","none",
    "null","true","false","patient","doctor","hospital"
}

def tokenize_content(text):
    """Lowercase, remove punctuation, filter stopwords — keep content tokens."""
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 2]

def factual_consistency_score(prediction, source_note):
    """
    Token-level faithfulness: proportion of content tokens in the
    prediction that also appear in the source note.
    Score = |pred_tokens ∩ source_tokens| / |pred_tokens|
    Lexical overlap only; it does not establish factual correctness.
    """
    if not prediction or not source_note:
        return 0.0
    pred_tokens   = tokenize_content(prediction)
    source_tokens = set(tokenize_content(source_note))
    if not pred_tokens:
        return 0.0
    overlap = sum(1 for t in pred_tokens if t in source_tokens)
    return overlap / len(pred_tokens)

def compute_factual_consistency(predictions, source_notes):
    """Average factual consistency across all samples."""
    scores = []
    for pred, src in zip(predictions, source_notes):
        try:
            scores.append(factual_consistency_score(pred, src))
        except:
            continue
    return np.mean(scores) if scores else 0.0

# Use test_notes as the source for factual grounding
source_notes_eval = test_notes[:N_EVAL]

ext_fcs = compute_factual_consistency(extractive_preds,  source_notes_eval)
pre_fcs = compute_factual_consistency(pretrained_preds,  source_notes_eval)
ft_fcs  = compute_factual_consistency(finetuned_preds,   source_notes_eval)
print(f"  Extractive Source-token overlap:       {ext_fcs:.4f}")
print(f"  Pre-trained BART Source-token overlap: {pre_fcs:.4f}")
print(f"  Fine-tuned BART Source-token overlap:  {ft_fcs:.4f}")
print("Done.")

In [ ]:
# ── METRIC 5: Lexicon concept F1 ───────────────────────────────────────────────
# MEDCON measures overlap of clinical concepts between prediction and
# reference summary. We use a curated UMLS-inspired clinical concept
# lexicon covering: diseases, symptoms, medications, procedures,
# anatomy, lab tests, and clinical findings.
# Score = F1 of shared concept mentions (precision × recall based)
print("Computing Lexicon concept F1s...")

# Curated clinical concept lexicon (UMLS-inspired, domain-specific)
CLINICAL_CONCEPTS = {
    "diseases_conditions": [
        "hypertension","diabetes","cancer","tumor","fracture","infection",
        "pneumonia","asthma","arthritis","anemia","sepsis","stroke",
        "myocardial infarction","heart failure","renal failure","fibrosis",
        "necrosis","edema","stenosis","thrombosis","embolism","hemorrhage",
        "osteoporosis","osteomalacia","dysplasia","malignant","benign",
        "carcinoma","lymphoma","leukemia","disorder","syndrome","disease"
    ],
    "symptoms": [
        "pain","fever","nausea","vomiting","fatigue","weakness","dyspnea",
        "cough","headache","dizziness","swelling","inflammation","rash",
        "bleeding","bruising","tenderness","stiffness","numbness","paralysis",
        "seizure","confusion","loss of consciousness","shortness of breath"
    ],
    "medications": [
        "mg","dose","tablet","capsule","injection","antibiotic","analgesic",
        "steroid","insulin","aspirin","metformin","lisinopril","atorvastatin",
        "amoxicillin","ibuprofen","paracetamol","chemotherapy","immunotherapy",
        "medication","drug","prescription","therapy","treatment"
    ],
    "procedures": [
        "surgery","biopsy","resection","excision","transplant","amputation",
        "catheterization","dialysis","transfusion","ventilation","intubation",
        "anesthesia","laparoscopy","thoracotomy","arthroplasty","osteotomy"
    ],
    "diagnostics": [
        "mri","ct scan","x-ray","ultrasound","biopsy","blood test","ecg",
        "echocardiography","colonoscopy","endoscopy","spirometry",
        "hemoglobin","glucose","creatinine","platelet","white blood cell",
        "red blood cell","culture","urinalysis"
    ],
    "anatomy": [
        "heart","lung","liver","kidney","brain","spine","femur","tibia",
        "humerus","pelvis","abdomen","thorax","chest","neck","hip","knee",
        "elbow","shoulder","artery","vein","nerve","muscle","bone","joint"
    ]
}

# Flatten all concepts into a single set for fast lookup
ALL_CONCEPTS = set()
for category_concepts in CLINICAL_CONCEPTS.values():
    ALL_CONCEPTS.update(category_concepts)

def extract_concepts(text):
    """Extract clinical concepts present in text (case-insensitive substring match)."""
    if not text:
        return set()
    text_lower = text.lower()
    return {concept for concept in ALL_CONCEPTS if concept in text_lower}

def medcon_score_single(prediction, reference):
    """
    Compute custom lexicon F1 for a single prediction-reference pair.
    Precision = |pred_concepts ∩ ref_concepts| / |pred_concepts|
    Recall    = |pred_concepts ∩ ref_concepts| / |ref_concepts|
    F1        = 2 * P * R / (P + R)
    """
    pred_concepts = extract_concepts(prediction)
    ref_concepts  = extract_concepts(reference)
    if not pred_concepts or not ref_concepts:
        return 0.0
    overlap   = len(pred_concepts & ref_concepts)
    precision = overlap / len(pred_concepts)
    recall    = overlap / len(ref_concepts)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def compute_medcon(predictions, references):
    """Average custom lexicon F1 across all samples."""
    scores = []
    for pred, ref in zip(predictions, references):
        try:
            scores.append(medcon_score_single(pred, ref))
        except:
            continue
    return np.mean(scores) if scores else 0.0

ext_medcon = compute_medcon(extractive_preds, extractive_refs)
pre_medcon = compute_medcon(pretrained_preds, pretrained_refs)
ft_medcon  = compute_medcon(finetuned_preds,  finetuned_refs)
print(f"  Extractive MEDCON:       {ext_medcon:.4f}")
print(f"  Pre-trained BART MEDCON: {pre_medcon:.4f}")
print(f"  Fine-tuned BART MEDCON:  {ft_medcon:.4f}")
print("Done.")

In [ ]:
# ── Build complete results table ─────────────────────────────────────────
full_results_df = pd.DataFrame([
    {
        "Model": "Extractive (TF-IDF)",
        "ROUGE-1": ext_rouge["ROUGE-1"], "ROUGE-2": ext_rouge["ROUGE-2"], "ROUGE-L": ext_rouge["ROUGE-L"],
        "BERTScore-F1": ext_bs, "METEOR": ext_meteor,
        "Keyword preservation": ext_ner,
        "Source-token overlap": ext_fcs,
        "Lexicon concept F1": ext_medcon
    },
    {
        "Model": "Pre-trained BART",
        "ROUGE-1": pre_rouge["ROUGE-1"], "ROUGE-2": pre_rouge["ROUGE-2"], "ROUGE-L": pre_rouge["ROUGE-L"],
        "BERTScore-F1": pre_bs, "METEOR": pre_meteor,
        "Keyword preservation": pre_ner,
        "Source-token overlap": pre_fcs,
        "Lexicon concept F1": pre_medcon
    },
    {
        "Model": "Fine-tuned BART",
        "ROUGE-1": ft_rouge["ROUGE-1"], "ROUGE-2": ft_rouge["ROUGE-2"], "ROUGE-L": ft_rouge["ROUGE-L"],
        "BERTScore-F1": ft_bs, "METEOR": ft_meteor,
        "Keyword preservation": ft_ner,
        "Source-token overlap": ft_fcs,
        "Lexicon concept F1": ft_medcon
    }
])

metric_cols = [c for c in full_results_df.columns if c != "Model"]
for c in metric_cols:
    full_results_df[c] = full_results_df[c].round(4)

print("\n" + "=" * 110)
print("COMPLETE EVALUATION RESULTS")
print("=" * 110)
print(full_results_df.to_string(index=False))
print("=" * 110)


## Section 10 — Results Visualization

In [ ]:
# Plot 1: ROUGE comparison
models  = full_results_df["Model"].tolist()
colors  = ["#4e79a7", "#f28e2b", "#59a14f"]
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
x = np.arange(len(metrics))
bw = 0.22
offsets = [-bw, 0, bw]

fig, ax = plt.subplots(figsize=(12, 6))
for model_name, color, offset in zip(models, colors, offsets):
    vals = [full_results_df.loc[full_results_df["Model"]==model_name, m].values[0] for m in metrics]
    bars = ax.bar(x + offset, vals, width=bw, label=model_name, color=color, alpha=0.88, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f"{v:.4f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=13)
ax.set_ylabel("F-Score", fontsize=12)
ax.set_title("ROUGE Score Comparison: Extractive vs Pre-trained BART vs Fine-tuned BART",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "rouge_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rouge_comparison_chart.png")

In [ ]:
# Plot 2: Full metrics grouped bar — all 8 metrics
all_metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore-F1", "METEOR",
               "Keyword preservation", "Source-token overlap", "Lexicon concept F1"]
x2 = np.arange(len(all_metrics))
bw2 = 0.25
offsets2 = [-bw2, 0, bw2]

fig, ax = plt.subplots(figsize=(18, 7))
for model_name, color, offset in zip(models, colors, offsets2):
    vals = [full_results_df.loc[full_results_df["Model"]==model_name, m].values[0] for m in all_metrics]
    bars = ax.bar(x2 + offset, vals, width=bw2, label=model_name, color=color, alpha=0.88, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold", rotation=45)

ax.set_xticks(x2)
ax.set_xticklabels(all_metrics, fontsize=10, rotation=15, ha="right")
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Complete Evaluation: All Metrics Comparison",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "full_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: full_metrics_comparison.png")


## Section 11 — Qualitative Comparison

In [ ]:
for idx in range(3):
    print("\n" + "#"*80)
    print(f"# TEST SAMPLE {idx+1}")
    print("#"*80)
    print("\n>>> INPUT NOTE (first 400 chars):")
    print("-"*60)
    print(test_notes[idx][:400])
    print("\n>>> EXTRACTIVE (TF-IDF):")
    print("-"*60)
    print(extractive_preds[idx][:400] if idx < len(extractive_preds) else "N/A")
    print("\n>>> PRE-TRAINED BART:")
    print("-"*60)
    print(pretrained_preds[idx][:400] if idx < len(pretrained_preds) else "N/A")
    print("\n>>> FINE-TUNED BART:")
    print("-"*60)
    print(finetuned_preds[idx][:600] if idx < len(finetuned_preds) else "N/A")
    print("\n>>> REFERENCE SUMMARY:")
    print("-"*60)
    print(test_summaries[idx][:600])

## Section 12 — Inference Function

In [ ]:
def generate_clinical_summary(note: str) -> str:
    """
    Generate a structured clinical summary from a raw clinical note.
    Uses fine-tuned BART-large-cnn with beam search.
    """
    if not isinstance(note, str) or len(note.strip()) == 0:
        return "Error: Input note is empty or invalid."
    if finetuned_pipeline is None:
        return "Error: Fine-tuned pipeline not loaded."
    try:
        result = finetuned_pipeline(
            note, max_length=384, min_length=80,
            do_sample=False, truncation=True,
            num_beams=4, length_penalty=1.2,
            no_repeat_ngram_size=3
        )
        return result[0]["summary_text"]
    except Exception as e:
        return f"Generation error: {str(e)}"

# Demo
demo_note   = test_notes[5]
demo_output = generate_clinical_summary(demo_note)
print("=" * 70)
print("DEMO: generate_clinical_summary()")
print("=" * 70)
print("INPUT (first 600 chars):"); print(demo_note[:600])
print("\nOUTPUT:"); print(demo_output)
print("=" * 70)

## Section 13 — Final Summary

In [ ]:
print("Experiment complete. Results from this run:")
print(full_results_df.to_string(index=False))
full_results_df.to_csv(ARTIFACT_DIR / "run_metrics.csv", index=False)
print("NER extractor:", "SciSpaCy" if SCISPACY_AVAILABLE else "regex keyword fallback")
print("These scores do not establish clinical correctness or suitability for patient care.")
print("The baseline and fine-tuned inference paths use different input/decoding settings.")
print("JSON-style output is not schema-validated.")
print("Files saved under:", ARTIFACT_DIR)
